# TRABALHO: DATA QUALITY

## Testes de Qualidade de Dados

### Introdução

Apresentamos neste trabalho um conjunto básico de testes essenciais, focando em ausência de dados (nulls), conformidade de esquema, volume e intervalos de valores. O desafio propõe que os ALUNOS evoluam para testes mais específicos, como validação de valores (finitos, numéricos, de data/hora e de formato), verificação de unicidade e testes de integridade referencial, para obter alertas informativos e contextuais. Finalmente, estes testes poderiam ser incorporados ao Pipeline de Dados para garantia de qualidade.

## ** ATENÇÃO **

Este notebook serve apenas um propósito educativo.

In [ ]:
# Em seguida iremos importar diversas bibliotecas que serão utilizadas:

# Pacote para trabalhar com JSON
import json

# Pacote para realizar requisições HTTP
import requests

# Pacote para exploração e análise de dados
import pandas as pd

# Pacote com métodos numéricos e representações matriciais
import numpy as np

# Pacotes do scikit-learn para pré-processamento de dados
# "SimpleImputer" é uma transformação para preencher valores faltantes em conjuntos de dados
from sklearn.impute import SimpleImputer

# Pacotes do scikit-learn para treinamento de modelos e construção de pipelines
# Método para separação de conjunto de dados em amostras de treino e teste
from sklearn.model_selection import train_test_split
# Método para criação de modelos baseados em árvores de decisão
from sklearn.tree import DecisionTreeClassifier
# Classe para a criação de uma pipeline de machine-learning
from sklearn.pipeline import Pipeline

# Pacotes do scikit-learn para avaliação de modelos
# Métodos para validação cruzada do modelo criado
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import accuracy_score

In [ ]:
# LE O DATASET COMO UM PANDAS DATAFRAME.
# Certifique-se de que o arquivo 'curso.csv' está no mesmo diretório que este notebook,
# ou forneça o caminho completo para o arquivo.
try:
    df_data_1 = pd.read_csv('curso.txt')
except FileNotFoundError:
    print("Arquivo 'curso.csv' não encontrado. Verifique o caminho e o nome do arquivo.")
    # Você pode querer interromper a execução ou carregar dados de exemplo aqui
    # Por exemplo, criar um DataFrame vazio para evitar erros subsequentes:
    df_data_1 = pd.DataFrame()

if not df_data_1.empty:
    print("Dados originais carregados:")
    print(df_data_1.head())
    print("\nInformações iniciais do DataFrame:")
    df_data_1.info()

Temos 15 colunas presentes no dataset fornecido, sendo a maioria delas variáveis características (dados de entrada) e uma delas uma variável-alvo (que queremos que o nosso modelo seja capaz de prever). 

As variáveis características são:

    MATRICULA          - número de matrícula do estudante
    NOME               - nome completo do estudante
    REPROVACOES_MAT_1  - número de reprovações na disciplina MAT_1
    REPROVACOES_MAT_2  - número de reprovações na disciplina MAT_2
    REPROVACOES_MAT_3  - número de reprovações na disciplina MAT_3
    REPROVACOES_MAT_4  - número de reprovações na disciplina MAT_4
    NOTA_MAT_1         - média simples das notas do aluno na disciplina MAT_1 (0-10)
    NOTA_MAT_2         - média simples das notas do aluno na disciplina MAT_2 (0-10)
    NOTA_MAT_3         - média simples das notas do aluno na disciplina MAT_3 (0-10)
    NOTA_MAT_4         - média simples das notas do aluno na disciplina MAT_4 (0-10)
    INGLES             - indica se o estudante tem conhecimento em língua inglesa (0 -> não, 1 -> sim, NaN -> não informado)
    H_AULA_PRES        - horas de estudo presencial realizadas pelo estudante
    TAREFAS_ONLINE     - número de tarefas online entregues pelo estudante
    FALTAS             - número de faltas acumuladas do estudante (todas disciplinas)
    
A variável-alvo é:

    PERFIL               - uma *string* que indica uma de cinco possibilidades: 
        "EXCELENTE"      - Estudante não necessita de mentoria
        "MUITO BOM"      - Estudante não necessita de mentoria
        "BOM"            - Estudante não necessita de mentoria
        "REGULAR"        - Estudante necessita de mentoria em algumas matérias
        "DIFICULDADE"    - Estudante necessita de mentoria em várias disciplinas
        
Com um modelo capaz de classificar um estudante em uma dessas categorias, podemos automatizar parte da mentoria estudantil através de assistentes virtuais, que serão capazes de recomendar práticas de estudo e conteúdo personalizado com base nas necessidades de cada aluno.

## TESTES DE QUALIDADE INICIAIS (Após Carga)

# ALUNOS: completar as células abaixo com testes para os dados brutos (df_data_1).

--------------------------------------------------------------------------------
**Testes de Qualidade dos Dados - Etapa Inicial (Após Carga em `df_data_1`)**

**a) TESTES DE SCHEMA**

* **Objetivo:** Verificar se as colunas esperadas estão presentes, com os nomes e tipos de dados corretos. Mudanças de schema são problemas comuns.
* **Instrução para o Aluno:** Verifiquem se as colunas originais do arquivo `curso.csv` foram carregadas corretamente em `df_data_1` e se os tipos de dados inferidos pelo pandas fazem sentido para cada coluna.

In [ ]:
# a) TESTES DE SCHEMA — df_data_1
if not df_data_1.empty:
    print('=== SCHEMA TESTS ===')

    # Colunas esperadas e seus tipos pandas
    expected_schema = {
        'MATRICULA':         'int64',
        'NOME':              'object',
        'REPROVACOES_MAT_1': 'int64',
        'REPROVACOES_MAT_2': 'int64',
        'REPROVACOES_MAT_3': 'int64',
        'REPROVACOES_MAT_4': 'int64',
        'NOTA_MAT_1':        'float64',
        'NOTA_MAT_2':        'float64',
        'NOTA_MAT_3':        'float64',
        'NOTA_MAT_4':        'float64',
        'INGLES':            'float64',
        'H_AULA_PRES':       'int64',
        'TAREFAS_ONLINE':    'int64',
        'FALTAS':            'int64',
        'PERFIL':            'object',
    }

    # Verifica colunas presentes
    missing_cols = [c for c in expected_schema if c not in df_data_1.columns]
    extra_cols   = [c for c in df_data_1.columns if c not in expected_schema]
    assert len(missing_cols) == 0, f'FALHA — Colunas ausentes: {missing_cols}'
    print(f'  [OK] Todas as {len(expected_schema)} colunas esperadas estão presentes.')

    if extra_cols:
        print(f'  [AVISO] Colunas extras encontradas: {extra_cols}')

    # Verifica tipos de dados
    type_errors = []
    for col, expected_dtype in expected_schema.items():
        actual_dtype = str(df_data_1[col].dtype)
        if actual_dtype != expected_dtype:
            type_errors.append(f'{col}: esperado={expected_dtype}, encontrado={actual_dtype}')

    if type_errors:
        print(f'  [AVISO] Divergências de tipo (podem ocorrer devido a NaN):')
        for e in type_errors:
            print(f'          {e}')
    else:
        print('  [OK] Todos os tipos de dados estão conforme o esperado.')

    print('\nTipos encontrados:')
    print(df_data_1.dtypes)
else:
    print('[SKIP] df_data_1 está vazio — testes de schema não executados.')


**b) TESTES DE VOLUME**

* **Objetivo:** Verificar se a quantidade de dados (número de linhas) está dentro do esperado. Importante para monitorar ingestão e custos.
* **Instrução para o Aluno:** Verifiquem o número total de linhas no DataFrame `df_data_1` carregado. Comparem com o número esperado de registros (se souberem) ou apenas verifiquem se não está vazio e se tem um volume mínimo razoável.

In [ ]:
# b) TESTES DE VOLUME — df_data_1
if not df_data_1.empty:
    print('=== VOLUME TESTS ===')

    n_rows = len(df_data_1)
    n_cols = len(df_data_1.columns)
    MIN_ROWS_EXPECTED = 10
    EXPECTED_COLS     = 15

    assert n_rows > 0, 'FALHA — DataFrame está vazio (0 linhas).'
    print(f'  [OK] DataFrame não está vazio: {n_rows} linhas encontradas.')

    assert n_rows >= MIN_ROWS_EXPECTED, (
        f'FALHA — Volume abaixo do mínimo: {n_rows} < {MIN_ROWS_EXPECTED} registros.'
    )
    print(f'  [OK] Volume mínimo atingido: {n_rows} >= {MIN_ROWS_EXPECTED} registros.')

    assert n_cols == EXPECTED_COLS, (
        f'FALHA — Número de colunas incorreto: encontrado={n_cols}, esperado={EXPECTED_COLS}.'
    )
    print(f'  [OK] Número de colunas correto: {n_cols} colunas.')

    # Percentual de nulos por coluna
    null_pct = (df_data_1.isnull().sum() / n_rows * 100).round(2)
    high_null = null_pct[null_pct > 20]
    if not high_null.empty:
        print(f'  [AVISO] Colunas com mais de 20% de nulos:\n{high_null}')
    else:
        print('  [OK] Nenhuma coluna com mais de 20% de valores nulos.')

    print(f'\n  Resumo: {n_rows} linhas x {n_cols} colunas')
    print(f'  Nulos por coluna:\n{df_data_1.isnull().sum()}')
else:
    print('[SKIP] df_data_1 está vazio — testes de volume não executados.')


**c) TESTES DE VALORES (Domínio)**

* **Objetivo:** Verificar se os valores em colunas categóricas ou de domínio restrito pertencem a um conjunto finito de valores esperados. Diferenciar de "corretude".
* **Instrução para o Aluno:** Verifiquem se a coluna `PERFIL` em `df_data_1` contém apenas os valores descritos no enunciado ("EXCELENTE", "MUITO BOM", "BOM", "REGULAR", "DIFICULDADE"). Verifiquem os valores possíveis na coluna `INGLES` antes da transformação (deve ser 0.0, 1.0 ou NaN).

In [ ]:
# c) TESTES DE VALORES (Domínio) — df_data_1
if not df_data_1.empty:
    print('=== VALORES (DOMÍNIO) TESTS ===')

    # PERFIL: conjunto de valores válidos (dataset usa MUITO_BOM com underscore)
    perfis_validos = {'EXCELENTE', 'MUITO_BOM', 'BOM', 'REGULAR', 'DIFICULDADE'}
    perfis_encontrados = set(df_data_1['PERFIL'].dropna().unique())
    perfis_invalidos   = perfis_encontrados - perfis_validos

    assert len(perfis_invalidos) == 0, (
        f'FALHA — Valores inválidos em PERFIL: {perfis_invalidos}'
    )
    print(f'  [OK] PERFIL contém apenas valores válidos: {sorted(perfis_encontrados)}')

    # INGLES: valores válidos são 0.0, 1.0 ou NaN
    ingles_validos = {0.0, 1.0}
    ingles_encontrados = set(df_data_1['INGLES'].dropna().unique())
    ingles_invalidos   = ingles_encontrados - ingles_validos

    assert len(ingles_invalidos) == 0, (
        f'FALHA — Valores inválidos em INGLES: {ingles_invalidos}'
    )
    n_ingles_nan = df_data_1['INGLES'].isnull().sum()
    print(f'  [OK] INGLES contém apenas valores válidos (0.0, 1.0 ou NaN).')
    print(f'       Valores encontrados: {sorted(ingles_encontrados)} | NaN: {n_ingles_nan}')

    # REPROVACOES: devem ser inteiros >= 0
    for mat in range(1, 5):
        col = f'REPROVACOES_MAT_{mat}'
        invalidos = df_data_1[df_data_1[col] < 0]
        assert len(invalidos) == 0, f'FALHA — Valores negativos em {col}: {len(invalidos)} registros.'
    print('  [OK] Todas as colunas REPROVACOES_MAT_X contêm apenas valores não-negativos.')

    print('\n  Contagem de valores em PERFIL:')
    print(df_data_1['PERFIL'].value_counts())
else:
    print('[SKIP] df_data_1 está vazio — testes de valores não executados.')


**d) TESTES NUMÉRICOS E DATAS**

* **Objetivo:** Verificar se valores numéricos estão dentro de ranges esperados (mínimo, máximo, etc.) e se datas (se houvesse) estão em formatos e ranges válidos.
* **Instrução para o Aluno:** Verifiquem se as notas (`NOTA_MAT_X`) em `df_data_1` estão dentro de um range razoável (0-10, considerando que antes do tratamento podem ter NaNs). Verifiquem se `REPROVACOES_MAT_X`, `H_AULA_PRES`, `TAREFAS_ONLINE`, `FALTAS` são não-negativos.

In [ ]:
# d) TESTES NUMÉRICOS E DATAS — df_data_1
if not df_data_1.empty:
    print('=== TESTES NUMÉRICOS ===')

    # Notas: range esperado [0.0, 10.0]
    for mat in range(1, 5):
        col = f'NOTA_MAT_{mat}'
        notas_validas = df_data_1[col].dropna()
        abaixo_min = (notas_validas < 0).sum()
        acima_max  = (notas_validas > 10).sum()
        assert abaixo_min == 0, f'FALHA — {col} possui {abaixo_min} valores < 0.'
        assert acima_max  == 0, f'FALHA — {col} possui {acima_max} valores > 10.'
        print(f'  [OK] {col}: min={notas_validas.min():.1f}, max={notas_validas.max():.1f} — dentro do range [0, 10].')

    # Colunas não-negativas
    cols_nao_negativas = [
        'REPROVACOES_MAT_1', 'REPROVACOES_MAT_2', 'REPROVACOES_MAT_3', 'REPROVACOES_MAT_4',
        'H_AULA_PRES', 'TAREFAS_ONLINE', 'FALTAS'
    ]
    for col in cols_nao_negativas:
        valores = df_data_1[col].dropna()
        negativos = (valores < 0).sum()
        assert negativos == 0, f'FALHA — {col} possui {negativos} valores negativos.'
    print(f'  [OK] Colunas não-negativas sem valores negativos: {cols_nao_negativas}')

    # Estatísticas descritivas das notas
    notas_cols = [f'NOTA_MAT_{i}' for i in range(1, 5)]
    print('\n  Estatísticas das notas:')
    print(df_data_1[notas_cols].describe().round(2))
else:
    print('[SKIP] df_data_1 está vazio — testes numéricos não executados.')


**e) TESTES DE FORMATOS**

* **Objetivo:** Verificar se os valores em colunas de texto ou identificadores seguem um formato ou padrão esperado.
* **Instrução para o Aluno:** Verifiquem se a coluna `MATRICULA` em `df_data_1` contém apenas valores que parecem ser numéricos (o tipo já foi checado no Schema, aqui pode-se pensar em comprimento ou padrão se fosse string). Verifiquem se a coluna `NOME` não está vazia ou contém apenas espaços.

In [ ]:
# e) TESTES DE FORMATOS — df_data_1
import re

if not df_data_1.empty:
    print('=== TESTES DE FORMATO ===')

    # MATRICULA: deve ser numérico (inteiro positivo de 6 dígitos)
    matriculas = df_data_1['MATRICULA'].dropna()
    nao_positivos = (matriculas <= 0).sum()
    assert nao_positivos == 0, f'FALHA — MATRICULA possui {nao_positivos} valores <= 0.'
    print(f'  [OK] MATRICULA: todos os valores são inteiros positivos.')

    # MATRICULA: comprimento esperado (6 dígitos)
    formato_invalido = df_data_1['MATRICULA'].apply(lambda x: len(str(int(x))) != 6 if pd.notna(x) else False)
    n_formato_inv = formato_invalido.sum()
    if n_formato_inv > 0:
        print(f'  [AVISO] MATRICULA: {n_formato_inv} valores não possuem 6 dígitos.')
        print(df_data_1.loc[formato_invalido, 'MATRICULA'].head())
    else:
        print('  [OK] MATRICULA: todos os valores possuem 6 dígitos.')

    # NOME: não deve ser vazio ou apenas espaços
    nomes_invalidos = df_data_1['NOME'].apply(
        lambda x: pd.isna(x) or str(x).strip() == ''
    )
    n_nomes_inv = nomes_invalidos.sum()
    assert n_nomes_inv == 0, f'FALHA — NOME possui {n_nomes_inv} registros vazios ou apenas espaços.'
    print(f'  [OK] NOME: todos os {len(df_data_1)} registros têm nome preenchido.')

    # NOME: deve conter pelo menos um espaço (nome e sobrenome)
    nomes_sem_sobrenome = df_data_1['NOME'].apply(
        lambda x: ' ' not in str(x).strip() if pd.notna(x) else False
    )
    n_sem_sob = nomes_sem_sobrenome.sum()
    if n_sem_sob > 0:
        print(f'  [AVISO] NOME: {n_sem_sob} registros sem sobrenome (sem espaço).')
    else:
        print('  [OK] NOME: todos os registros contêm nome e sobrenome.')

    print(f'\n  Exemplos de MATRICULA: {sorted(df_data_1["MATRICULA"].head().tolist())}')
    print(f'  Exemplos de NOME: {df_data_1["NOME"].head(3).tolist()}')
else:
    print('[SKIP] df_data_1 está vazio — testes de formato não executados.')


**f) TESTES DE UNICIDADE**

* **Objetivo:** Verificar se colunas que deveriam ter valores únicos (como chaves primárias) de fato não contêm duplicados.
* **Instrução para o Aluno:** Verifiquem se a coluna `MATRICULA` em `df_data_1` contém apenas valores únicos, pois ela provavelmente serve como identificador de cada estudante.

In [ ]:
# f) TESTES DE UNICIDADE — df_data_1
if not df_data_1.empty:
    print('=== TESTES DE UNICIDADE ===')

    # MATRICULA deve ser chave primária: sem duplicatas
    n_total    = len(df_data_1)
    n_unicos   = df_data_1['MATRICULA'].nunique()
    n_dupl     = n_total - n_unicos

    assert n_dupl == 0, (
        f'FALHA — MATRICULA possui {n_dupl} registros duplicados.'
    )
    print(f'  [OK] MATRICULA é única: {n_unicos} valores distintos em {n_total} registros.')

    # Verifica duplicatas no par (MATRICULA, NOME)
    dup_mat_nome = df_data_1.duplicated(subset=['MATRICULA', 'NOME']).sum()
    assert dup_mat_nome == 0, (
        f'FALHA — Par (MATRICULA, NOME) possui {dup_mat_nome} duplicatas.'
    )
    print(f'  [OK] Par (MATRICULA, NOME): sem duplicatas.')

    # Verifica se NOME tem valores repetidos (pode ser esperado — alunos homônimos)
    nomes_dup = df_data_1['NOME'].duplicated().sum()
    if nomes_dup > 0:
        print(f'  [AVISO] NOME possui {nomes_dup} valores repetidos (possível homônimo).')
        print(df_data_1[df_data_1['NOME'].duplicated(keep=False)][['MATRICULA', 'NOME']].head())
    else:
        print('  [OK] NOME: sem valores repetidos.')

    print(f'\n  is_unique MATRICULA: {df_data_1["MATRICULA"].is_unique}')
else:
    print('[SKIP] df_data_1 está vazio — testes de unicidade não executados.')


**g) TESTES DE INTEGRIDADE REFERENCIAL (Intra-Tabela)**

* **Objetivo:** Verificar relacionamentos lógicos entre colunas dentro da mesma tabela. Em uma única tabela, verifica consistência entre campos relacionados.
* **Instrução para o Aluno:** Como este é um dataset de tabela única, verifiquem a integridade entre colunas relacionadas em `df_data_1`. Por exemplo, se um aluno tem reprovações em uma matéria (`REPROVACOES_MAT_X > 0`), é esperado que a nota correspondente (`NOTA_MAT_X`) seja baixa (por exemplo, menor que 4, conforme a lógica futura do notebook sugere para "REPROVADO"). Ou o contrário, se a nota é alta, não deveria ter reprovações. **Importante:** Este teste pode ser complexo antes das transformações, pois NaN e 0.0 (para quem não cursou) impactam a lógica. Exemplo de uma regra simples: se `NOTA_MAT_1` é < 4 (e não NaN), `REPROVACOES_MAT_1` deveria ser > 0.

In [ ]:
# g) TESTES DE INTEGRIDADE REFERENCIAL (Intra-Tabela) — df_data_1
if not df_data_1.empty:
    print('=== INTEGRIDADE REFERENCIAL — df_data_1 ===')

    violacoes_total = 0

    for mat in range(1, 5):
        nota_col = f'NOTA_MAT_{mat}'
        reprov_col = f'REPROVACOES_MAT_{mat}'

        # Regra: se nota < 4 (e não é NaN e não é 0 — 0 = ainda não cursou)
        # então REPROVACOES_MAT deve ser > 0
        mask_nota_valida = df_data_1[nota_col].notna() & (df_data_1[nota_col] > 0)
        df_cursou = df_data_1[mask_nota_valida]

        # Se nota < 4, espera-se ao menos 1 reprovação
        violacao_reprov = df_cursou[
            (df_cursou[nota_col] < 4) & (df_cursou[reprov_col] == 0)
        ]
        n_viol = len(violacao_reprov)
        violacoes_total += n_viol
        if n_viol > 0:
            print(f'  [AVISO] {nota_col} < 4 mas {reprov_col} == 0: {n_viol} registro(s).')
            print(violacao_reprov[[nota_col, reprov_col, 'PERFIL']].head())
        else:
            print(f'  [OK] {nota_col}/{reprov_col}: consistência nota-reprovação verificada.')

        # Regra inversa: se nota >= 4 e reprovações > 0 — logicamente inconsistente
        violacao_nota_alta = df_data_1[
            df_data_1[nota_col].notna() &
            (df_data_1[nota_col] >= 4) &
            (df_data_1[reprov_col] > 0)
        ]
        n_viol2 = len(violacao_nota_alta)
        violacoes_total += n_viol2
        if n_viol2 > 0:
            print(f'  [AVISO] {nota_col} >= 4 mas {reprov_col} > 0: {n_viol2} registro(s) inconsistentes.')
            print(violacao_nota_alta[[nota_col, reprov_col, 'PERFIL']].head())
        else:
            print(f'  [OK] {nota_col} >= 4 nunca ocorre com {reprov_col} > 0.')

    print(f'\n  Total de violações de integridade encontradas: {violacoes_total}')
else:
    print('[SKIP] df_data_1 está vazio — testes de integridade não executados.')


--------------------------------------------------------------------------------
### Transformação 1: Tratando dados faltantes para `NOTA_MAT_4` e `INGLES`

Para tratar os dados faltantes em nosso conjunto de dados, iremos agora utilizar uma transformação pronta da biblioteca scikit-learn, chamada **SimpleImputer**, e o método `fillna()` do pandas.

Neste exemplo:
1.  Os valores faltantes em `INGLES` serão preenchidos com `-1` (indicando 'SEM RESPOSTA'). Isso cria `df_data_2`.
2.  Os valores faltantes em `NOTA_MAT_4` serão preenchidos com `0` usando `SimpleImputer`. Isso atualiza `df_data_2` para `df_data_3`.

In [ ]:
if not df_data_1.empty:
    print("Valores nulos em df_data_1 ANTES do tratamento de NaN: \n{}".format(df_data_1.isnull().sum()))

    # Etapa 1.1: Tratar INGLES e criar df_data_2
    # df_data_2 será o DataFrame após o preenchimento de NaNs em 'INGLES'
    df_data_2 = df_data_1.copy()
    df_data_2["INGLES"].fillna(-1, inplace=True)
    print("\nValores nulos em df_data_2 (APÓS tratar INGLES com -1): \n{}".format(df_data_2.isnull().sum()))

    # Etapa 1.2: Tratar NOTA_MAT_4 em df_data_2 para criar df_data_3
    # df_data_3 será o DataFrame após a imputação de NaNs em 'NOTA_MAT_4' (e já com 'INGLES' tratado)
    df_data_3 = df_data_2.copy()
    imputer_nota_mat_4 = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)
    df_data_3[['NOTA_MAT_4']] = imputer_nota_mat_4.fit_transform(df_data_3[['NOTA_MAT_4']])
    # Nota: Se a intenção do notebook original era imputar TODOS os NaNs numéricos com 0,
    # o SimpleImputer deveria ser aplicado de forma mais ampla. Aqui, focamos em NOTA_MAT_4.
    
    print("\n--- Após Tratamento Inicial de Dados Faltantes --- ")
    print("Valores nulos em df_data_3 (APÓS tratar INGLES com -1 e NOTA_MAT_4 com 0): \n{}".format(df_data_3.isnull().sum()))
    # O print original do prompt usava df_data_2.isnull().sum() aqui. 
    # Se df_data_2 é o estado *após* INGLES e NOTA_MAT_4 tratados, então o print deve usar o df final dessa etapa (df_data_3).
    # Se df_data_2 no print do prompt era para mostrar o estado *antes* de NOTA_MAT_4, então o código acima está correto
    # em criar df_data_2 e df_data_3 como estados sequenciais para os testes.
else:
    print("DataFrame df_data_1 está vazio. Transformações de NaN não podem ser executadas.")
    # Define df_data_2 e df_data_3 como vazios para evitar erros nas células de teste
    df_data_2 = pd.DataFrame()
    df_data_3 = pd.DataFrame()

## TESTES DE QUALIDADE (Após Tratamento Inicial de Dados Faltantes)

# ALUNOS: completar as células abaixo com testes para verificar se a transformação de dados faltantes funcionou como esperado.
# Testes para 'INGLES' usarão `df_data_2`.
# Testes para 'NOTA_MAT_4' usarão `df_data_3`.

--------------------------------------------------------------------------------
**Testes de Qualidade dos Dados - Etapa 2 (Após Tratamento de Dados Faltantes)**

**c) TESTES DE VALORES (Revisão)**

* **Objetivo:** Verificar se os valores após o tratamento de NaN estão conforme o esperado (e.g., -1 para `INGLES`).
* **Instrução para o Aluno:** Verifiquem se a coluna `INGLES` em `df_data_2` agora inclui o valor -1.0 e não tem mais NaNs.

In [ ]:
# c) TESTES DE VALORES (Revisão) — df_data_2: INGLES tratado
if not df_data_2.empty:
    print('=== VALORES (REVISÃO) — df_data_2: INGLES ===')

    # Sem NaN em INGLES
    n_nan_ingles = df_data_2['INGLES'].isnull().sum()
    assert n_nan_ingles == 0, f'FALHA — INGLES ainda possui {n_nan_ingles} NaN(s) após tratamento.'
    print(f'  [OK] INGLES: nenhum NaN após preenchimento com -1. ({n_nan_ingles} NaNs)')

    # -1 deve estar presente (indica 'SEM RESPOSTA')
    ingles_values = set(df_data_2['INGLES'].unique())
    assert -1 in ingles_values or -1.0 in ingles_values, (
        'FALHA — Valor -1 não encontrado em INGLES após tratamento (esperado para NaN original).'
    )
    print(f'  [OK] INGLES: valor -1 presente (NaN substituído). Valores distintos: {sorted(ingles_values)}')

    # Apenas valores válidos pós-tratamento: {-1, 0, 1}
    ingles_validos_pos = {-1.0, 0.0, 1.0}
    invalidos = ingles_values - ingles_validos_pos
    assert len(invalidos) == 0, f'FALHA — Valores inesperados em INGLES: {invalidos}'
    print(f'  [OK] INGLES: apenas valores {sorted(ingles_validos_pos)} após tratamento.')

    print(f'\n  Contagem de INGLES em df_data_2:\n{df_data_2["INGLES"].value_counts(dropna=False)}')
else:
    print('[SKIP] df_data_2 está vazio — teste de valores não executado.')


**d) TESTES NUMÉRICOS E DATAS (Revisão)**

* **Objetivo:** Verificar se os ranges ou valores específicos após o tratamento de NaN estão conforme o esperado (e.g., 0 para `NOTA_MAT_4` onde era NaN).
* **Instrução para o Aluno:** Verifiquem se a coluna `NOTA_MAT_4` em `df_data_3` não tem mais NaNs e agora inclui o valor 0 onde antes era NaN. Verifiquem o valor mínimo da coluna `NOTA_MAT_4`.

In [ ]:
# d) TESTES NUMÉRICOS (Revisão) — df_data_3: NOTA_MAT_4 tratada
if not df_data_3.empty:
    print('=== TESTES NUMÉRICOS (REVISÃO) — df_data_3: NOTA_MAT_4 ===')

    # Sem NaN em NOTA_MAT_4
    n_nan = df_data_3['NOTA_MAT_4'].isnull().sum()
    assert n_nan == 0, f'FALHA — NOTA_MAT_4 ainda possui {n_nan} NaN(s) após imputação.'
    print(f'  [OK] NOTA_MAT_4: nenhum NaN após imputação com 0. ({n_nan} NaNs)')

    # Valor mínimo deve ser 0 (NaN virou 0)
    min_nota = df_data_3['NOTA_MAT_4'].min()
    assert min_nota == 0.0, f'FALHA — NOTA_MAT_4 mínimo esperado 0.0, encontrado {min_nota}.'
    print(f'  [OK] NOTA_MAT_4: valor mínimo = {min_nota} (NaN substituído por 0).')

    # Range ainda dentro de [0, 10]
    max_nota = df_data_3['NOTA_MAT_4'].max()
    assert max_nota <= 10.0, f'FALHA — NOTA_MAT_4 máximo fora do range: {max_nota} > 10.'
    print(f'  [OK] NOTA_MAT_4: valor máximo = {max_nota} <= 10.')

    # Confirmar que o numero de linhas com NOTA_MAT_4 == 0 é >= ao de NaN original em df_data_1
    n_zeros   = (df_data_3['NOTA_MAT_4'] == 0.0).sum()
    n_nan_orig = df_data_1['NOTA_MAT_4'].isnull().sum() if not df_data_1.empty else 0
    assert n_zeros >= n_nan_orig, (
        f'FALHA — Esperado ao menos {n_nan_orig} zeros em NOTA_MAT_4, encontrado {n_zeros}.'
    )
    print(f'  [OK] NOTA_MAT_4: {n_zeros} zeros presentes (>= {n_nan_orig} NaN(s) originais imputados).')

    print(f'\n  Estatísticas NOTA_MAT_4 em df_data_3:\n{df_data_3["NOTA_MAT_4"].describe().round(2)}')
else:
    print('[SKIP] df_data_3 está vazio — teste numérico não executado.')


Nota-se que não deveriamos ter mais nenhum valor faltante nas colunas `INGLES` e `NOTA_MAT_4` após essas transformações específicas.

### Transformação 2: Criação de novas colunas descritivas (`CURSOU_MAT_X_DESC`) e imputação da média nas notas para casos de 'AINDA NAO CURSOU'

In [ ]:
# A entrada para esta transformação é df_data_3
if not df_data_3.empty:
    df = df_data_3.copy() # Usar uma cópia para criar df_data_4

    # 4) Criar colunas CURSOU_MATX_DESC
    # APROVADO = 1, NOTA >= 4 e REPROVACOES == 0
    cond1_mat1 = (df['NOTA_MAT_1'] >= 4) & (df['REPROVACOES_MAT_1'] == 0)
    cond1_mat2 = (df['NOTA_MAT_2'] >= 4) & (df['REPROVACOES_MAT_2'] == 0)
    cond1_mat3 = (df['NOTA_MAT_3'] >= 4) & (df['REPROVACOES_MAT_3'] == 0)
    cond1_mat4 = (df['NOTA_MAT_4'] >= 4) & (df['REPROVACOES_MAT_4'] == 0)
    
    # REPROVADO = 0, NOTA < 4 e REPROVACOES > 0 (ou NOTA < 4 mesmo com REPROVACOES == 0 se for a primeira vez)
    # A lógica original do notebook era: (df['NOTA_MAT_X'] < 4) & (df['REPROVACOES_MAT_X'] > 0)
    # Vamos manter essa lógica. Casos como nota < 4 e reprovações == 0 não seriam classificados por esta regra nem pela de aprovado.
    # Eles seriam 'AINDA NAO CURSOU' se NOTA == 0 e REPROVACOES == 0.
    # Se NOTA_MAT_X é < 4 mas > 0 e REPROVACOES_MAT_X == 0, ficaria sem categoria por estas duas condições.
    # O np.select usará o valor default (0) se nenhuma condição for atendida, ou o último da lista de 'choices'.
    # Para ser mais explícito, a regra de REPROVADO pode ser apenas NOTA < 4 (assumindo que se cursou e não aprovou, é reprovado)
    # No entanto, o notebook original tem uma lógica específica para 'AINDA NAO CURSOU' (NOTA == 0 e REPROVACOES == 0)
    # Vamos seguir a lógica do notebook original para CURSOU_MATX_DESC, que parece ser:
    # 1 (APROVADO), 0 (REPROVADO), -1 (AINDA NAO CURSOU)

    # REPROVADO = 0
    cond2_mat1 = (df['NOTA_MAT_1'] < 4) & (df['REPROVACOES_MAT_1'] > 0) # Lógica original do notebook
    # Uma lógica mais abrangente para reprovado poderia ser apenas (df['NOTA_MAT_1'] < 4) & (df['NOTA_MAT_1'] != 0)
    # Mas isso conflitaria com a definição de 'AINDA NAO CURSOU' se NOTA_MAT_1=0 e REPROVACOES_MAT_1 > 0 (que não deveria ocorrer)
    cond2_mat2 = (df['NOTA_MAT_2'] < 4) & (df['REPROVACOES_MAT_2'] > 0)
    cond2_mat3 = (df['NOTA_MAT_3'] < 4) & (df['REPROVACOES_MAT_3'] > 0)
    cond2_mat4 = (df['NOTA_MAT_4'] < 4) & (df['REPROVACOES_MAT_4'] > 0)

    # AINDA NAO CURSOU = -1 : NOTA = 0, SEM REPROVAÇÕES
    # Esta condição é crucial. NOTA_MAT_4 já foi imputada para 0 se era NaN.
    # Se NOTA_MAT_X == 0 E REPROVACOES_MAT_X == 0, então AINDA NAO CURSOU.
    cond3_mat1 = (df['NOTA_MAT_1'] == 0) & (df['REPROVACOES_MAT_1'] == 0)
    cond3_mat2 = (df['NOTA_MAT_2'] == 0) & (df['REPROVACOES_MAT_2'] == 0)
    cond3_mat3 = (df['NOTA_MAT_3'] == 0) & (df['REPROVACOES_MAT_3'] == 0)
    cond3_mat4 = (df['NOTA_MAT_4'] == 0) & (df['REPROVACOES_MAT_4'] == 0)

    conditions_MAT1 = [cond1_mat1, cond2_mat1, cond3_mat1]
    conditions_MAT2 = [cond1_mat2, cond2_mat2, cond3_mat2]
    conditions_MAT3 = [cond1_mat3, cond2_mat3, cond3_mat3]
    conditions_MAT4 = [cond1_mat4, cond2_mat4, cond3_mat4]

    choices = [1, 0, -1] # 1:APROVADO, 0:REPROVADO, -1:AINDA NAO CURSOU
    # O default do np.select é 0. Se uma linha não se encaixa em nenhuma condição, 
    # CURSOU_MATX_DESC será 0. Isso pode ser problemático se 0 é REPROVADO.
    # É importante que as condições cubram todos os casos ou que o default seja bem pensado.
    # Para este caso, se não for APROVADO, REPROVADO (pela regra estrita), ou AINDA NAO CURSOU,
    # o que seria? Ex: NOTA=3, REPROVACOES=0. Não é APROVADO, não é REPROVADO (pela regra), não é AINDA NAO CURSOU.
    # O ideal seria ter uma categoria 'OUTRO' ou ajustar as regras para serem exaustivas.
    # Por ora, vamos usar um default que não seja uma das categorias válidas, como -99, e depois verificar.
    default_choice = -99 # Um valor para identificar casos não cobertos pelas regras explícitas

    df['CURSOU_MAT1_DESC'] = np.select(conditions_MAT1, choices, default=default_choice)
    df['CURSOU_MAT2_DESC'] = np.select(conditions_MAT2, choices, default=default_choice)
    df['CURSOU_MAT3_DESC'] = np.select(conditions_MAT3, choices, default=default_choice)
    df['CURSOU_MAT4_DESC'] = np.select(conditions_MAT4, choices, default=default_choice)

    # Verificar se algum default foi usado
    for i in range(1,5):
        if default_choice in df[f'CURSOU_MAT{i}_DESC'].unique():
            print(f"ALERTA: Casos não cobertos pelas regras em CURSOU_MAT{i}_DESC. Valor {default_choice} atribuído.")
            # print(df[df[f'CURSOU_MAT{i}_DESC'] == default_choice][['NOME', f'NOTA_MAT_{i}', f'REPROVACOES_MAT_{i}']])
    
    # Imputação da média nas notas para 'AINDA NAO CURSOU'
    MEDIA_NOTA_MAT_1 = df.loc[df['CURSOU_MAT1_DESC'] != -1, 'NOTA_MAT_1'].mean()
    MEDIA_NOTA_MAT_2 = df.loc[df['CURSOU_MAT2_DESC'] != -1, 'NOTA_MAT_2'].mean()
    MEDIA_NOTA_MAT_3 = df.loc[df['CURSOU_MAT3_DESC'] != -1, 'NOTA_MAT_3'].mean()
    MEDIA_NOTA_MAT_4 = df.loc[df['CURSOU_MAT4_DESC'] != -1, 'NOTA_MAT_4'].mean()
    
    # Armazenar médias para referência nos testes
    medias_calculadas = {
        'MEDIA_NOTA_MAT_1': MEDIA_NOTA_MAT_1,
        'MEDIA_NOTA_MAT_2': MEDIA_NOTA_MAT_2,
        'MEDIA_NOTA_MAT_3': MEDIA_NOTA_MAT_3,
        'MEDIA_NOTA_MAT_4': MEDIA_NOTA_MAT_4
    }
    print("\nMédias calculadas (para quem cursou):")
    for key, value in medias_calculadas.items():
        print(f"  {key}: {value:.2f}")

    df.loc[df['CURSOU_MAT1_DESC'] == -1, 'NOTA_MAT_1'] = MEDIA_NOTA_MAT_1
    df.loc[df['CURSOU_MAT2_DESC'] == -1, 'NOTA_MAT_2'] = MEDIA_NOTA_MAT_2
    df.loc[df['CURSOU_MAT3_DESC'] == -1, 'NOTA_MAT_3'] = MEDIA_NOTA_MAT_3
    df.loc[df['CURSOU_MAT4_DESC'] == -1, 'NOTA_MAT_4'] = MEDIA_NOTA_MAT_4

    df_data_4 = df # DataFrame final após esta transformação

    print("\n--- Após Criação de Colunas Descritivas e Imputação de Médias (df_data_4) ---")
    print("Novas colunas adicionadas e médias imputadas.")
    print(df_data_4[['NOME', 'NOTA_MAT_1', 'REPROVACOES_MAT_1', 'CURSOU_MAT1_DESC', 
                     'NOTA_MAT_2', 'REPROVACOES_MAT_2', 'CURSOU_MAT2_DESC']].head())
else:
    print("DataFrame df_data_3 está vazio. Transformação 2 não pode ser executada.")
    df_data_4 = pd.DataFrame() # Define df_data_4 como vazio para evitar erros

## TESTES DE QUALIDADE (Após Criação de Colunas Descritivas e Imputação de Médias)

# ALUNOS: completar as células abaixo com testes para verificar as novas colunas (`CURSOU_MAT_X_DESC`) e os valores de notas imputados em `df_data_4`.

--------------------------------------------------------------------------------
**Testes de Qualidade dos Dados - Etapa Final (Após Criação de Colunas e Imputação de Médias em `df_data_4`)**

**e) TESTES DE VALORES (Novas Colunas)**

* **Objetivo:** Verificar se os valores nas novas colunas criadas (`CURSOU_MAT_X_DESC`) estão conforme o esperado (1, 0, -1, ou o valor default se usado).
* **Instrução para o Aluno:** Verifiquem se as novas colunas `CURSOU_MAT_X_DESC` em `df_data_4` contêm apenas os valores 1, 0 ou -1.

In [ ]:
# e) TESTES DE VALORES (Novas Colunas) — df_data_4
if not df_data_4.empty:
    print('=== VALORES (NOVAS COLUNAS CURSOU_MAT_X_DESC) — df_data_4 ===')

    valores_validos_desc = {1, 0, -1}
    for mat in range(1, 5):
        col = f'CURSOU_MAT{mat}_DESC'
        valores_encontrados = set(df_data_4[col].unique())
        invalidos = valores_encontrados - valores_validos_desc

        if invalidos:
            print(f'  [FALHA] {col}: valores inesperados encontrados: {invalidos}')
            print(f'           (1=APROVADO, 0=REPROVADO, -1=AINDA NAO CURSOU)')
            # Mostrar os registros com valores inesperados
            mask_inv = df_data_4[col].isin(invalidos)
            print(df_data_4.loc[mask_inv, ['NOME', f'NOTA_MAT_{mat}', f'REPROVACOES_MAT_{mat}', col]].head())
        else:
            contagem = df_data_4[col].value_counts().to_dict()
            print(f'  [OK] {col}: valores válidos: {sorted(valores_encontrados)} | contagem: {contagem}')

    print('\n  Resumo das colunas descritivas:')
    desc_cols = [f'CURSOU_MAT{i}_DESC' for i in range(1, 5)]
    print(df_data_4[desc_cols].value_counts().head(10))
else:
    print('[SKIP] df_data_4 está vazio — teste de valores não executado.')


**f) TESTES NUMÉRICOS E DATAS (Após Imputação da Média)**

* **Objetivo:** Verificar se as médias foram calculadas e imputadas corretamente nas notas onde `CURSOU_MAT_X_DESC` é -1.
* **Instrução para o Aluno:** Verifiquem se os registros onde `CURSOU_MAT_1_DESC` é -1 (AINDA NAO CURSOU) em `df_data_4` agora têm o valor da `NOTA_MAT_1` igual à média calculada (`MEDIA_NOTA_MAT_1`). Repitam para as outras matérias.

In [ ]:
# f) TESTES NUMÉRICOS (Após Imputação da Média) — df_data_4
if not df_data_4.empty:
    print('=== TESTES NUMÉRICOS (IMPUTAÇÃO DE MÉDIA) — df_data_4 ===')

    for mat in range(1, 5):
        nota_col = f'NOTA_MAT_{mat}'
        desc_col = f'CURSOU_MAT{mat}_DESC'
        media_esperada = medias_calculadas.get(f'MEDIA_NOTA_MAT_{mat}')

        if media_esperada is None:
            print(f'  [SKIP] Média para MAT_{mat} não encontrada.')
            continue

        # Registros 'AINDA NAO CURSOU' (-1) devem ter nota == media_esperada
        df_nao_cursou = df_data_4[df_data_4[desc_col] == -1]
        n_nao_cursou  = len(df_nao_cursou)

        if n_nao_cursou == 0:
            print(f'  [OK] {nota_col}: nenhum aluno com {desc_col} == -1; imputação não aplicada.')
            continue

        # Verificar com tolerância para ponto flutuante
        notas_imputadas = df_nao_cursou[nota_col]
        nao_imputed = (~np.isclose(notas_imputadas, media_esperada, rtol=1e-5)).sum()

        assert nao_imputed == 0, (
            f'FALHA — {nota_col}: {nao_imputed} registro(s) com CURSOU_MAT{mat}_DESC==-1 '
            f'não têm a média imputada ({media_esperada:.4f}).'
        )
        print(f'  [OK] {nota_col}: {n_nao_cursou} aluno(s) com {desc_col}==-1 '
              f'têm nota imputada = {media_esperada:.4f}.')

        # Sem NaN nas notas após imputação
        n_nan = df_data_4[nota_col].isnull().sum()
        assert n_nan == 0, f'FALHA — {nota_col} ainda possui {n_nan} NaN(s) após imputação.'
        print(f'  [OK] {nota_col}: sem NaN após imputação de média.')

    print('\n  Estatísticas finais das notas em df_data_4:')
    notas_cols = [f'NOTA_MAT_{i}' for i in range(1, 5)]
    print(df_data_4[notas_cols].describe().round(2))
else:
    print('[SKIP] df_data_4 está vazio — teste numérico não executado.')


**g) TESTES DE INTEGRIDADE REFERENCIAL (Consistência Lógica Após Criação de Colunas)**

* **Objetivo:** Verificar se a lógica das novas colunas `CURSOU_MAT_X_DESC` (1: APROVADO, 0: REPROVADO, -1: AINDA NAO CURSOU) está consistente com os valores originais de `NOTA_MAT_X` e `REPROVACOES_MAT_X` em `df_data_4`, baseados nas condições definidas na transformação.
* **Instrução para o Aluno:** Verifiquem se não há registros onde a coluna `CURSOU_MAT_1_DESC` indica "APROVADO" (1), mas `NOTA_MAT_1` é menor que 4 ou `REPROVACOES_MAT_1` é maior que 0. Verifiquem as outras combinações lógicas definidas para REPROVADO (0) e AINDA NAO CURSOU (-1) para todas as matérias.

In [ ]:
# g) TESTES DE INTEGRIDADE REFERENCIAL (Consistência Lógica) — df_data_4
if not df_data_4.empty:
    print('=== INTEGRIDADE REFERENCIAL (CURSOU_MAT_X_DESC) — df_data_4 ===')

    for mat in range(1, 5):
        nota_col  = f'NOTA_MAT_{mat}'
        reprov_col = f'REPROVACOES_MAT_{mat}'
        desc_col   = f'CURSOU_MAT{mat}_DESC'
        media_esp  = medias_calculadas.get(f'MEDIA_NOTA_MAT_{mat}', 0)

        print(f'  --- MAT_{mat} ---')

        # APROVADO (1): nota_orig >= 4 E reprovacoes == 0
        # Nota: após imputação, alunos com desc==-1 têm nota == media; devemos verificar
        # a regra de APROVADO apenas para quem realmente cursou (desc != -1)
        df_aprov = df_data_4[df_data_4[desc_col] == 1]
        viol_aprov = df_aprov[
            (df_aprov[nota_col] < 4) | (df_aprov[reprov_col] > 0)
        ]
        assert len(viol_aprov) == 0, (
            f'FALHA — {desc_col}==1 (APROVADO) mas nota<4 ou reprovacoes>0: {len(viol_aprov)} caso(s).'
        )
        print(f'    [OK] APROVADO  (1): {len(df_aprov)} aluno(s) com nota>=4 e reprovacoes==0.')

        # REPROVADO (0): nota < 4 E reprovacoes > 0
        df_reprov = df_data_4[df_data_4[desc_col] == 0]
        viol_reprov = df_reprov[
            (df_reprov[nota_col] >= 4) | (df_reprov[reprov_col] == 0)
        ]
        assert len(viol_reprov) == 0, (
            f'FALHA — {desc_col}==0 (REPROVADO) mas nota>=4 ou reprovacoes==0: {len(viol_reprov)} caso(s).'
        )
        print(f'    [OK] REPROVADO (0): {len(df_reprov)} aluno(s) com nota<4 e reprovacoes>0.')

        # AINDA NAO CURSOU (-1): nota imputada == media E reprovacoes == 0
        df_nao_cursou = df_data_4[df_data_4[desc_col] == -1]
        viol_nao_curs = df_nao_cursou[
            (~np.isclose(df_nao_cursou[nota_col], media_esp, rtol=1e-5)) |
            (df_nao_cursou[reprov_col] != 0)
        ]
        assert len(viol_nao_curs) == 0, (
            f'FALHA — {desc_col}==-1 (AINDA NAO CURSOU) com nota != media ou reprovacoes != 0: '
            f'{len(viol_nao_curs)} caso(s).'
        )
        print(f'    [OK] NÃO CURSOU(-1): {len(df_nao_cursou)} aluno(s) com nota=media({media_esp:.2f}) e reprovacoes==0.')

    print('\n  Integridade referencial pós-transformação: OK')
else:
    print('[SKIP] df_data_4 está vazio — teste de integridade não executado.')


--------------------------------------------------------------------------------
### Treinando um modelo de classificação
O restante do notebook com o treino do modelo pode permanecer. É importante que os alunos reflitam sobre como a qualidade dos dados (verificada e melhorada nas etapas anteriores) impacta o desempenho e a confiabilidade do modelo de Machine Learning.

In [ ]:
if not df_data_4.empty:
    # Definição das colunas que serão features (nota-se que a coluna NOME não está presente)
    features = [
        #"MATRICULA", # Geralmente não é uma feature para o modelo em si
        "REPROVACOES_MAT_1", 'REPROVACOES_MAT_2', "REPROVACOES_MAT_3", "REPROVACOES_MAT_4",
        "NOTA_MAT_1", "NOTA_MAT_2", "NOTA_MAT_3", "NOTA_MAT_4",
        "INGLES", "H_AULA_PRES", "TAREFAS_ONLINE", "FALTAS", 
        # As colunas CURSOU_MATX_DESC são representações do estado do aluno, podem ou não ser usadas como features
        # Se usadas, as notas originais para 'AINDA NAO CURSOU' foram alteradas, o que deve ser considerado.
        # Para este exemplo, vamos usar as notas já tratadas e as reprovações.
        # Poderia ser interessante testar o modelo com e sem as colunas CURSOU_MATX_DESC como features.
        'CURSOU_MAT1_DESC', 'CURSOU_MAT2_DESC', 'CURSOU_MAT3_DESC', 'CURSOU_MAT4_DESC'
    ]

    # Definição da variável-alvo
    target = "PERFIL" # Target é uma string, não uma lista de strings para y

    # Preparação dos argumentos para os métodos da biblioteca ``scikit-learn``
    X = df_data_4[features]
    y = df_data_4[target]

    # Separação dos dados em um conjunto de treino e um conjunto de teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=337, stratify=y) # Adicionado stratify

    # Criação de uma árvore de decisão com a biblioteca ``scikit-learn``:
    decision_tree = DecisionTreeClassifier(random_state=337)

    # Treino do modelo (é chamado o método *fit()* com os conjuntos de treino)
    decision_tree.fit(X_train, y_train)

    # Realização de teste cego no modelo criado
    y_pred = decision_tree.predict(X_test)

    # Acurácia alcançada pela árvore de decisão
    acc = accuracy_score(y_test, y_pred)
    print(f"Acurácia do modelo DecisionTreeClassifier: {100*round(acc, 4)}%")
else:
    print("DataFrame df_data_4 está vazio. Treinamento do modelo não pode ser executado.")

#### SALVA RESULTADO TRANSFORMADO

In [ ]:
# Salva em CSV os dados preparados (df_data_4):
if not df_data_4.empty:
    try:
        df_data_4.to_csv("curso_ml_transformado.csv", encoding='utf-8-sig', index=False)
        print("\nDataFrame transformado (df_data_4) salvo em 'curso_ml_transformado.csv'")
    except Exception as e:
        print(f"Erro ao salvar o arquivo CSV: {e}")
else:
    print("DataFrame df_data_4 está vazio. Nada para salvar.")

### Itens que podem ser usados na correção

Outras sugestões de utilização:

* **Compreensão do Objetivo do Teste:** entender o propósito de cada tipo de teste (Schema, Volume, Valores, etc.)? A descrição ou os comentários no código refletem esse entendimento?
* **Aplicação ao Dataset:** aplicar o teste corretamente ao dataset `curso.csv` e às transformações específicas realizadas no notebook? Os exemplos escolhidos para testar (e.g., coluna `MATRICULA` para unicidade, `PERFIL` para valores) são relevantes para `df_data_1`, `df_data_2`, `df_data_3` e `df_data_4` conforme a etapa?
* **Correção da Lógica de Teste:** O código Python implementado está correto para realizar a verificação proposta? Ele identifica potenciais problemas nos dados? Foram usados `asserts` de forma eficaz?
* **Utilização de Ferramentas Adequadas:** utilizar métodos apropriados do pandas ou numpy para realizar as verificações (e.g., `.columns`, `.dtypes`, `len()`, `.unique()`, `.isin()`, `.min()`, `.max()`, `.is_unique`, `.isnull().sum()`, filtragem booleana, `np.isclose()`)?
* **Cobertura:** aplicar os testes em colunas ou aspectos relevantes dos dados originais e, crucialmente, nos dados transformados e nas novas colunas criadas? (Por exemplo, testar os valores possíveis de `CURSOU_MAT_X_DESC` e a consistência entre `NOTA_MAT_X`, `REPROVACOES_MAT_X` e `CURSOU_MAT_X_DESC` após a transformação).
* **Clareza do Código e Comentários:** deixar claro qual teste está sendo realizado e o que ele verifica?
* **Interpretação dos Resultados:** comentar no vídeo sobre os resultados dos testes, especialmente se alguma anomalia ou falha de asserção (se ativada) foi encontrada? Explicar o que um resultado inesperado significa?
* **Adaptação dos Exemplos:** Adaptar os exemplos e adicionando novos testes relevantes para as colunas e transformações específicas?
* **Tratamento de Erros/Casos Limites:** ajustar casos com dados vazios ou colunas não existentes ao escrever seus testes ?
* **Reflexão (Opcional):** identificar nuances ou ambiguidades na regra de negócio ou nos dados brutos (como a questão do valor 0 em `NOTA_MAT_X` que pode significar "não cursou" ou "reprovou com nota zero" antes das transformações explícitas) e ajustou o teste ou comentou sobre isso?